In [ ]:
import sys, subprocess, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark", "datasets", "mlflow", "transformers", "torch", "accelerate"])
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/amazon-reviews-sentiment-analysis")
else:
    sys.path.insert(0, os.path.abspath("..")) if os.path.basename(os.getcwd()) == "notebooks" else None

# DistilBERT Baseline for Sentiment Analysis

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from src.config import TransformerConfig, DataConfig
from src.data_loader import load_data_pandas

t_config = TransformerConfig()
d_config = DataConfig()

train_df, test_df = load_data_pandas(d_config)
train_df = train_df.sample(n=t_config.sample_size, random_state=42)
test_df = test_df.sample(n=t_config.sample_size // 4, random_state=42)  # 50K test

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["combined_text"].tolist(), train_df["label"].tolist(),
    test_size=0.1, random_state=42, stratify=train_df["label"]
)
print(f"Train: {len(train_texts):,} | Val: {len(val_texts):,} | Test: {len(test_df):,}")

In [ ]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained(t_config.model_name)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=t_config.max_length)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=t_config.max_length)
test_encodings = tokenizer(test_df["combined_text"].tolist(), truncation=True, padding=True, max_length=t_config.max_length)

In [ ]:
import torch
from torch.utils.data import Dataset

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)
test_dataset = ReviewDataset(test_encodings, test_df["label"].tolist())

In [ ]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_hf_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

model = DistilBertForSequenceClassification.from_pretrained(t_config.model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./transformer_results",
    num_train_epochs=t_config.epochs,
    per_device_train_batch_size=t_config.batch_size,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./transformer_logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    learning_rate=t_config.lr,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_hf_metrics,
)

trainer.train()

In [ ]:
test_results = trainer.evaluate(test_dataset)
print("DistilBERT Test Results:")
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## PySpark vs DistilBERT Comparison

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "Training Data", "Training Time"],
    "Best PySpark (LR+CV)": ["~0.9017", "~0.9017", "~0.9017", "~0.9017", "3.6M", "~5 min"],
    "DistilBERT": [
        f"{test_results.get('eval_accuracy', 'N/A'):.4f}",
        f"{test_results.get('eval_precision', 'N/A'):.4f}",
        f"{test_results.get('eval_recall', 'N/A'):.4f}",
        f"{test_results.get('eval_f1', 'N/A'):.4f}",
        f"{t_config.sample_size:,}",
        "GPU-dependent",
    ],
})
print(comparison.to_string(index=False))

In [ ]:
from src.experiment_tracker import init_mlflow, log_experiment

init_mlflow()
log_experiment(
    run_name="distilbert_baseline",
    model_name="distilbert",
    feature_type="transformer_embeddings",
    hyperparams={"max_length": t_config.max_length, "epochs": t_config.epochs, "lr": t_config.lr, "sample_size": t_config.sample_size},
    metrics={k.replace("eval_", ""): v for k, v in test_results.items() if isinstance(v, float)},
    training_time=0,
)
print("Results logged to MLflow.")